In [ ]:
# magic code
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from securerag.config import Config


cfg = Config()
cfg.device = "cpu"
cfg.n_context = 10
cfg.batch_size = 10 
cfg.private_passage_ratio = 0.5

In [ ]:
# Prepare Data
import torch
import transformers
from securerag import data

checkpoint_path = "models/nq_reader_base"

path = "data/open_domain_data/NQ/dev.json"
datas = data.load(path=path, size=cfg.load_size)
dataset = data.Dataset(data=datas, n_context=cfg.n_context)
tokenizer: transformers.T5Tokenizer = transformers.T5Tokenizer.from_pretrained(
    "models/t5-base", return_dict=False
)
data_loader = torch.utils.data.dataloader.DataLoader(
    dataset=dataset,
    batch_size=cfg.batch_size,
    collate_fn=data.SecureRAG4T5Collator(
        tokenizer=tokenizer,
        text_maxlength=cfg.text_maxlength,
        answer_maxlength=cfg.answer_maxlength,
        private_passage_ratio=cfg.private_passage_ratio,
    ),
)
record1 = next(iter(data_loader))

/root/miniconda3/envs/securerag/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1767: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
/root/miniconda3/envs/securerag/lib/python3.8/site-packages/transformers/tokenization_t5.py:184: UserWarning: This sequence already has </s>. In future versions this behavior may lead to duplicated eos tokens being added.
  warnings.warn(


In [ ]:
# Model
from securerag.models import FiDT5


model_cls = FiDT5
model_path = cfg.generator_model_path
model = model_cls.from_pretrained(model_path)
model = model.to(cfg.device)
model.eval()

#  convert to securerag from fidtt5
from securerag.models.securerag import SecureRAG

model: SecureRAG = SecureRAG(fidt5=model)

In [ ]:
# generate
import time


device = cfg.device
(
    question_ids,  # bsz * 1 * dim
    question_masks,
    context_ids,
    context_masks,  # bsz * docs * dim
    private_context_ids,
    private_context_masks,  # bsz * docs_p * dim
    scores,  # bsz * docs
    private_scores,
) = (
    record1.question_ids,
    record1.question_masks,
    record1.passage_ids,
    record1.passage_masks,
    record1.private_passage_ids,
    record1.private_passage_masks,
    record1.scores,
    record1.private_scores,
)

start = time.time()
output = model.generate(
    context_ids=context_ids,
    context_ids_private=private_context_ids,
    attention_mask=context_masks,
    attention_mask_private=private_context_masks,
    doc_scores=scores,
    doc_scores_private=private_scores,
    max_length=50,
)

ans = tokenizer.batch_decode(output, skip_special_tokens=True)
print(ans)
print(f"elapsed time : {time.time() - start: .3f} sec")

tensor([[False],
        [False],
        [False],
        [ True],
        [ True],
        [ True],
        [ True],
        [False],
        [False],
        [ True]])
['Linda Kaye Davis', '2,000', 'through the Saint Lawrence River', 'September 25, 2018', '239', 'Hayden Panettiere', 'Roy Larson Raymond', '18 May 2002', 'Charles Cornwallis', 'Atlantic Records']
elapsed time :  8.141 sec
